> **Методологическое уточнение.** Этот ноутбук сохраняет исторический эксперимент. Первоначальная DQN снизила условную стоимость, но дала 46 перегрузок против 4 у порогового правила. Q и DQN имеют разные представления состояния и бюджеты настройки. Исправленное сравнение и границы выводов: [ноутбук 10](10_methodology_audit.ipynb), [аудит](METHODOLOGY_REVIEW.md).

# 03 — Табличное Q-learning

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from simulator import *
from agents import *
from experiments import *
assert Path('data/processed.npz').exists(), 'Сначала выполните data_pipeline.py'
data=load_data()
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.2})


Q-learning запоминает оценки действий в таблице. Состояние дискретизировано: число шардов, прошлый максимум загрузки, знак тренда, время суток, корзина цены следующего расширения. Непосещённые состояния используют равные Q; детерминированный выбор первого действия означает −1, что при двух шардах сохраняет два.

Проверяем alpha 0.05/0.15 и gamma 0.90/0.97, каждый вариант с seed 7/42/123, 600 эпизодов train. Каждые 50 эпизодов оцениваем validation и сохраняем лучший checkpoint. Сначала выбираем гиперпараметры по средней validation cost трёх seed, затем seed по validation. Test не используется. Повторение одних четырёх дней может приводить к переобучению.

In [2]:
# Готовые результаты загружаются. Для повторного обучения поставьте retrain=True.
search=run_q_study(retrain=False)
display(search)
display(search.groupby(['alpha','gamma']).validation_cost.agg(['mean','std']))
print(Path('results/q/selection.json').read_text())

,name,alpha,gamma,seed,validation_cost
0,a0.05_g0.9_s7,0.05,0.90,7,2549.700991
1,a0.05_g0.9_s42,0.05,0.90,42,2466.220667
2,a0.05_g0.9_s123,0.05,0.90,123,2355.748411
3,a0.05_g0.97_s7,0.05,0.97,7,2467.985259
4,a0.05_g0.97_s42,0.05,0.97,42,2479.967727
5,a0.05_g0.97_s123,0.05,0.97,123,2365.875162
6,a0.15_g0.9_s7,0.15,0.90,7,2444.375104
7,a0.15_g0.9_s42,0.15,0.90,42,2430.341187
8,a0.15_g0.9_s123,0.15,0.90,123,2381.246159
9,a0.15_g0.97_s7,0.15,0.97,7,2546.568151


mean        std
alpha gamma                        
0.05  0.90   2457.223356  97.288820
      0.97   2437.942716  62.699236
0.15  0.90   2418.654150  33.147489
      0.97   2455.186489  81.377900

{
  "name": "a0.15_g0.9_s123",
  "alpha": 0.15,
  "gamma": 0.9,
  "seed": 123,
  "validation_cost": 2381.246159158245
}
